# Prompt 10: Window Functions in Spark SQL and DataFrame API
## Databricks Certified Associate Developer for Apache Spark
### Topic 2 — Using Spark SQL (20%)

---

## What Are Window Functions?

A **window function** performs a calculation across a set of rows **related to the current row** —
without collapsing those rows into a single output row (unlike `groupBy().agg()`).

```
Regular aggregation (groupBy):       Window function:
┌────────┬────────┬────────┐         ┌────────┬────────┬────────┬─────────┐
│ dept   │ emp    │ salary │         │ dept   │ emp    │ salary │ avg_sal │
├────────┼────────┼────────┤  →      ├────────┼────────┼────────┼─────────┤
│ Eng    │  95000 │        │  1 row  │ Eng    │ Alice  │  95000 │  99,500 │  ← row kept!
└────────┴────────┴────────┘         │ Eng    │ Carol  │ 110000 │  99,500 │
                                     │ Eng    │ James  │  93500 │  99,500 │
                                     └────────┴────────┴────────┴─────────┘
```

**Key difference:** `groupBy().agg()` → N groups → N rows. Window function → every input row is preserved.

---

## WindowSpec — The Core Building Block

```python
from pyspark.sql.window import Window

# Define the window specification
windowSpec = (
    Window
    .partitionBy('department')          # PARTITION BY — like GROUP BY but keeps rows
    .orderBy('salary')                  # ORDER BY — required for ranking/offset functions
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)  # frame boundary
)

# Apply the window function
df.withColumn('running_total', F.sum('salary').over(windowSpec))
```

### Frame Boundaries

| Boundary Constant | Meaning |
|------------------|---------|
| `Window.unboundedPreceding` | First row of the partition |
| `Window.currentRow` | The current row |
| `Window.unboundedFollowing` | Last row of the partition |
| `n` (integer) | n rows before/after current row |

### rowsBetween vs rangeBetween

| Method | Frame type | Boundary unit |
|--------|-----------|---------------|
| `rowsBetween(start, end)` | Physical rows | Row count |
| `rangeBetween(start, end)` | Logical range | Value range based on ORDER BY column |

```python
# Last 3 rows + current row (physical rows)
Window.orderBy('date').rowsBetween(-3, 0)

# All rows in the partition (unbounded)
Window.partitionBy('dept').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# Running total from start of partition to current row
Window.partitionBy('dept').orderBy('date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, round as F_round
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('WindowFunctions') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Employee dataset
employees = spark.createDataFrame([
    (1,  'Alice',  'Engineering', 95000,  'London',   '2019-03-15'),
    (2,  'Bob',    'Marketing',   72000,  'New York',  '2018-06-01'),
    (3,  'Carol',  'Engineering', 110000, 'London',   '2016-01-10'),
    (4,  'Dave',   'HR',          65000,  'Chicago',  '2021-09-20'),
    (5,  'Eve',    'Engineering', 105000, 'Berlin',   '2017-11-05'),
    (6,  'Frank',  'Marketing',   78000,  'New York',  '2020-04-18'),
    (7,  'Grace',  'HR',          68000,  'Chicago',  '2019-07-30'),
    (8,  'Henry',  'Engineering', 95000,  'London',   '2022-02-14'),  # tie with Alice
    (9,  'Iris',   'Marketing',   82000,  'Berlin',   '2018-12-22'),
    (10, 'James',  'Engineering', 120000, 'New York',  '2015-08-08'),
], ['id', 'name', 'department', 'salary', 'city', 'hire_date'])

# Sales dataset for running totals / moving averages
sales = spark.createDataFrame([
    ('2024-01-01', 'East',  12000),
    ('2024-01-02', 'East',   8500),
    ('2024-01-03', 'East',  15000),
    ('2024-01-04', 'East',   9200),
    ('2024-01-05', 'East',  11000),
    ('2024-01-01', 'West',   7000),
    ('2024-01-02', 'West',  13000),
    ('2024-01-03', 'West',   6500),
    ('2024-01-04', 'West',  14000),
    ('2024-01-05', 'West',   9800),
], ['sale_date', 'region', 'amount'])

employees.createOrReplaceTempView('employees')
sales.createOrReplaceTempView('sales')

print('Employees:')
employees.show()
print('Sales:')
sales.show()

## 1. Ranking Functions

| Function | Tied values behaviour | Gaps in sequence? |
|----------|----------------------|-------------------|
| `rank()` | Same rank for ties | **Yes** — next rank skips (1,1,3) |
| `dense_rank()` | Same rank for ties | **No** — consecutive (1,1,2) |
| `row_number()` | Unique number per row | N/A — always sequential (1,2,3) |
| `ntile(n)` | Divides partition into n equal buckets | N/A |
| `percent_rank()` | `(rank - 1) / (total_rows - 1)` | Ranges from 0.0 to 1.0 |

**Exam critical:** Given tied salaries of 95000 for rank position 2 in a group of 4:
- `rank()` → 2, 2, 4 (skips 3)
- `dense_rank()` → 2, 2, 3 (no gap)
- `row_number()` → 2, 3, 4 (arbitrary tiebreak — order within tie is non-deterministic)

In [ ]:
# Cell 2: Ranking functions — rank vs dense_rank vs row_number
from pyspark.sql.functions import rank, dense_rank, row_number, ntile, percent_rank

# Window: partition by department, order by salary descending
rank_window = Window.partitionBy('department').orderBy(col('salary').desc())

ranked = employees.withColumn('rank',          rank().over(rank_window)) \
                  .withColumn('dense_rank',    dense_rank().over(rank_window)) \
                  .withColumn('row_number',    row_number().over(rank_window)) \
                  .withColumn('ntile_2',       ntile(2).over(rank_window)) \
                  .withColumn('percent_rank',  F_round(percent_rank().over(rank_window), 2))

print('=== Ranking within department (sorted by salary desc) ===')
ranked.select('department', 'name', 'salary', 'rank', 'dense_rank', 'row_number', 'ntile_2', 'percent_rank') \
      .orderBy('department', col('salary').desc()) \
      .show()

# Key observation: Alice and Henry both have 95000 in Engineering
print('=== FOCUS: Tied salaries in Engineering ===')
ranked.filter(col('department') == 'Engineering') \
      .select('name', 'salary', 'rank', 'dense_rank', 'row_number') \
      .orderBy(col('salary').desc()) \
      .show()
# Expected:
# James  120000  rank=1  dense_rank=1  row_number=1
# Carol  110000  rank=2  dense_rank=2  row_number=2
# Eve    105000  rank=3  dense_rank=3  row_number=3
# Alice   95000  rank=4  dense_rank=4  row_number=4  ← tied
# Henry   95000  rank=4  dense_rank=4  row_number=5  ← tied (row_number is unique)
print('Note: Alice and Henry tied at 95000 — rank/dense_rank same, row_number unique')

In [ ]:
# Cell 3: Ranking functions in SQL syntax
print('=== SQL: RANK / DENSE_RANK / ROW_NUMBER / NTILE / PERCENT_RANK ===')
spark.sql("""
    SELECT
        department,
        name,
        salary,
        RANK()         OVER (PARTITION BY department ORDER BY salary DESC) AS rank,
        DENSE_RANK()   OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rank,
        ROW_NUMBER()   OVER (PARTITION BY department ORDER BY salary DESC) AS row_number,
        NTILE(2)       OVER (PARTITION BY department ORDER BY salary DESC) AS ntile_2,
        ROUND(PERCENT_RANK() OVER (PARTITION BY department ORDER BY salary DESC), 2) AS pct_rank
    FROM employees
    ORDER BY department, salary DESC
""").show()

# Top-1 per department using row_number
print('=== Top earner per department using ROW_NUMBER ===')
spark.sql("""
    SELECT department, name, salary
    FROM (
        SELECT department, name, salary,
               ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn
        FROM employees
    ) ranked
    WHERE rn = 1
    ORDER BY salary DESC
""").show()

## 2. Offset / Analytic Functions — lag() and lead()

| Function | Description |
|----------|-------------|
| `lag(col, offset, default)` | Value from `offset` rows **before** current row |
| `lead(col, offset, default)` | Value from `offset` rows **after** current row |
| `first(col, ignorenulls)` | First value in the window frame |
| `last(col, ignorenulls)` | Last value in the window frame |

**Default offset = 1.** Default value for out-of-bounds = NULL unless specified.

```python
lag(col('amount'), 1, 0)   # previous row amount, 0 if no previous row
lead(col('amount'), 2)     # amount 2 rows ahead, NULL if no such row
```

**Common use cases:**
- Day-over-day change: `col - lag(col, 1)`
- Time since last event: `datediff(current_date, lag(event_date, 1))`
- Next appointment: `lead(appointment_date, 1)`

In [ ]:
# Cell 4: lag() and lead() — offset functions
from pyspark.sql.functions import lag, lead, to_date

sales_dt = sales.withColumn('sale_date', to_date(col('sale_date'), 'yyyy-MM-dd'))

# Window: partition by region, order by date
time_window = Window.partitionBy('region').orderBy('sale_date')

lag_lead = sales_dt \
    .withColumn('prev_amount',    lag('amount', 1, 0).over(time_window)) \
    .withColumn('next_amount',    lead('amount', 1).over(time_window)) \
    .withColumn('day_over_day',   col('amount') - col('prev_amount')) \
    .withColumn('pct_change',     F_round(
        (col('amount') - col('prev_amount')) / col('prev_amount') * 100, 1
    ))

print('=== lag / lead — day-over-day change ===')
lag_lead.select('region', 'sale_date', 'amount', 'prev_amount', 'next_amount', 'day_over_day', 'pct_change') \
        .orderBy('region', 'sale_date') \
        .show()

# SQL equivalent
sales_dt.createOrReplaceTempView('sales_dt')
print('=== SQL: LAG / LEAD ===')
spark.sql("""
    SELECT
        region,
        sale_date,
        amount,
        LAG(amount, 1, 0) OVER (PARTITION BY region ORDER BY sale_date) AS prev_amount,
        LEAD(amount, 1)   OVER (PARTITION BY region ORDER BY sale_date) AS next_amount,
        amount - LAG(amount, 1, 0) OVER (PARTITION BY region ORDER BY sale_date) AS day_change
    FROM sales_dt
    ORDER BY region, sale_date
""").show()

## 3. Aggregate Window Functions — Running Totals & Moving Averages

Standard aggregate functions (`sum`, `avg`, `min`, `max`, `count`) become **window functions**
when used with `.over(windowSpec)`. They compute the aggregate **across the window frame**
for each row without reducing the row count.

### Common Frame Patterns

```python
# Running total (cumulative sum from start to current row)
Window.partitionBy('region').orderBy('date') \
      .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Moving average (last 3 rows + current row = 4-row window)
Window.partitionBy('region').orderBy('date') \
      .rowsBetween(-3, Window.currentRow)

# Partition-wide total (same value for all rows in partition)
Window.partitionBy('region') \
      .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

# Expanding window (all previous rows, no current — exclude current)
Window.partitionBy('region').orderBy('date') \
      .rowsBetween(Window.unboundedPreceding, -1)
```

In [ ]:
# Cell 5: Aggregate window functions — running totals, moving averages
from pyspark.sql.functions import sum as F_sum, avg as F_avg, min as F_min, max as F_max, count as F_count

# Running total window
running_window = Window.partitionBy('region') \
                       .orderBy('sale_date') \
                       .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Moving 3-day average window (current + 2 preceding)
moving_window = Window.partitionBy('region') \
                      .orderBy('sale_date') \
                      .rowsBetween(-2, Window.currentRow)

# Partition-total window (no ORDER BY needed for partition-wide)
total_window = Window.partitionBy('region') \
                     .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

agg_result = sales_dt \
    .withColumn('running_total',  F_sum('amount').over(running_window)) \
    .withColumn('moving_avg_3d',  F_round(F_avg('amount').over(moving_window), 0)) \
    .withColumn('region_total',   F_sum('amount').over(total_window)) \
    .withColumn('pct_of_total',   F_round(col('amount') / F_sum('amount').over(total_window) * 100, 1)) \
    .withColumn('running_min',    F_min('amount').over(running_window)) \
    .withColumn('running_max',    F_max('amount').over(running_window))

print('=== Running totals, moving averages, % of total ===')
agg_result.select(
    'region', 'sale_date', 'amount',
    'running_total', 'moving_avg_3d',
    'region_total', 'pct_of_total',
    'running_min', 'running_max'
).orderBy('region', 'sale_date').show()

# SQL equivalent
print('=== SQL: SUM/AVG as window functions ===')
spark.sql("""
    SELECT
        region,
        sale_date,
        amount,
        SUM(amount) OVER (
            PARTITION BY region ORDER BY sale_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total,
        ROUND(AVG(amount) OVER (
            PARTITION BY region ORDER BY sale_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 0) AS moving_avg_3d,
        SUM(amount) OVER (PARTITION BY region) AS region_total
    FROM sales_dt
    ORDER BY region, sale_date
""").show()

In [ ]:
# Cell 6: first() / last() over a window
from pyspark.sql.functions import first, last

# Show first and last sale amount in each region's partition
bounded_window = Window.partitionBy('region') \
                       .orderBy('sale_date') \
                       .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

print('=== first() / last() over unbounded window ===')
sales_dt.withColumn('first_sale',  first('amount').over(bounded_window)) \
        .withColumn('last_sale',   last('amount').over(bounded_window)) \
        .orderBy('region', 'sale_date') \
        .show()

# first() with ignoreNulls
nulls_data = spark.createDataFrame([
    ('A', 1, None),
    ('A', 2, 100),
    ('A', 3, 200),
    ('A', 4, None),
    ('A', 5, 300),
], ['grp', 'pos', 'val'])

w = Window.partitionBy('grp').orderBy('pos') \
          .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

print('=== first() with and without ignoreNulls ===')
nulls_data.withColumn('first_default',       first('val').over(w)) \
          .withColumn('first_ignore_nulls',  first('val', ignorenulls=True).over(w)) \
          .withColumn('last_default',        last('val').over(w)) \
          .withColumn('last_ignore_nulls',   last('val', ignorenulls=True).over(w)) \
          .show()
# first_default = null (first row is null)
# first_ignore_nulls = 100 (skips null)

In [ ]:
# Cell 7: Common exam patterns — Top-N per group, running total, compare to group avg
print('=== Pattern 1: Top-2 earners per department (row_number) ===')
from pyspark.sql.functions import row_number

w_dept_sal = Window.partitionBy('department').orderBy(col('salary').desc())

employees.withColumn('rn', row_number().over(w_dept_sal)) \
         .filter(col('rn') <= 2) \
         .select('department', 'name', 'salary', 'rn') \
         .orderBy('department', 'rn') \
         .show()

print('=== Pattern 2: Salary vs department average (% above/below) ===')
w_dept_all = Window.partitionBy('department') \
                   .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

employees.withColumn('dept_avg',    F_round(F_avg('salary').over(w_dept_all), 0)) \
         .withColumn('vs_avg_pct',  F_round(
             (col('salary') - F_avg('salary').over(w_dept_all)) /
              F_avg('salary').over(w_dept_all) * 100, 1
         )) \
         .select('department', 'name', 'salary', 'dept_avg', 'vs_avg_pct') \
         .orderBy('department', col('salary').desc()) \
         .show()

print('=== Pattern 3: Assign hire order within department (row_number by hire_date) ===')
from pyspark.sql.functions import to_date

w_hire = Window.partitionBy('department').orderBy('hire_date')
employees.withColumn('hire_order', row_number().over(w_hire)) \
         .select('department', 'hire_order', 'name', 'hire_date', 'salary') \
         .orderBy('department', 'hire_order') \
         .show()

In [ ]:
# Cell 8: rangeBetween vs rowsBetween
print('=== rowsBetween vs rangeBetween — understanding the difference ===')

data = spark.createDataFrame([
    ('A', 1, 100),
    ('A', 2, 200),
    ('A', 2, 300),  # same ORDER BY value as previous — rangeBetween groups them
    ('A', 3, 400),
    ('A', 4, 500),
], ['grp', 'seq', 'val'])

rows_w  = Window.partitionBy('grp').orderBy('seq') \
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)
range_w = Window.partitionBy('grp').orderBy('seq') \
                .rangeBetween(Window.unboundedPreceding, Window.currentRow)

data.withColumn('rows_running_sum',  F_sum('val').over(rows_w)) \
    .withColumn('range_running_sum', F_sum('val').over(range_w)) \
    .show()

# Expected difference at rows where seq=2:
# rowsBetween: physical row position — row 1 cumsum=100, row 2 cumsum=300, row 3 cumsum=600
# rangeBetween: VALUE-based — both seq=2 rows get sum including ALL seq<=2 rows = 600

print('Key insight:')
print('  rowsBetween: frame is physical row count — each row has its own frame')
print('  rangeBetween: frame is value-based on ORDER BY column — tied values share same frame')
print()
print('  rowsBetween(-2, 0) → include exactly 2 prior rows + current row')
print('  rangeBetween(-100, 0) → include all rows where ORDER BY col is within -100 of current')

spark.stop()
print('Done.')

## 4. Window Function Syntax Summary

### DataFrame API
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number, ntile, percent_rank
from pyspark.sql.functions import lag, lead, first, last
from pyspark.sql.functions import sum, avg, min, max, count

# 1. Define window specification
w = Window.partitionBy('dept').orderBy(col('salary').desc())

# 2. Apply with .over()
df.withColumn('rnk', rank().over(w))
df.withColumn('prev_sal', lag('salary', 1).over(w))
df.withColumn('running', F.sum('salary').over(
    w.rowsBetween(Window.unboundedPreceding, Window.currentRow)
))
```

### SQL Syntax
```sql
function_name() OVER (
    PARTITION BY col1, col2    -- optional: split into independent groups
    ORDER BY col3 ASC/DESC     -- required for ranking/offset; optional for aggregates
    ROWS BETWEEN start AND end -- optional: frame specification
)

-- Examples:
RANK() OVER (PARTITION BY dept ORDER BY salary DESC)
LAG(salary, 1, 0) OVER (PARTITION BY dept ORDER BY hire_date)
SUM(amount) OVER (PARTITION BY region ORDER BY sale_date
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
AVG(amount) OVER (PARTITION BY region ORDER BY sale_date
                  ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
```

### When to Use Window Functions vs groupBy

| Use case | Use | Reason |
|----------|-----|--------|
| Sum sales per region (one row per region) | `groupBy().agg()` | Row reduction needed |
| Running total per region (all rows kept) | Window function | No row reduction |
| Top-N per group | Window `row_number()` + `filter()` | Preserves all original columns |
| Salary vs group average | Window `avg()` + original rows | Need side-by-side comparison |
| Day-over-day change | Window `lag()` | Access to adjacent rows |
| Percentile ranking | `percent_rank()` / `ntile()` | Relative position needed |

## 5. Real-World Scenarios

<details>
<summary>Scenario 1: Ranking Employees by Salary Within Department for Promotion Decisions</summary>

**Situation:**
HR needs a ranked list of employees within each department by salary to identify
promotion candidates (top 25% by ntile) and flag salary compression
(employees within 5% of the next rank up).

**Code:**
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, ntile, lag, col, abs as F_abs, round as F_round

w = Window.partitionBy('department').orderBy(col('salary').desc())

hr_report = employees \
    .withColumn('salary_rank',  dense_rank().over(w)) \
    .withColumn('salary_bucket', ntile(4).over(w)) \
    .withColumn('next_salary',  lag('salary', -1).over(w)) \
    .withColumn('gap_to_next',  F_round(
        (col('salary') - col('next_salary')) / col('next_salary') * 100, 1
    )) \
    .withColumn('promotion_candidate', col('salary_bucket') == 1)

hr_report.select(
    'department', 'name', 'salary', 'salary_rank',
    'salary_bucket', 'gap_to_next', 'promotion_candidate'
).orderBy('department', 'salary_rank').show()
```

**Expected Outcome:**
Each employee has a rank, quartile bucket, and gap to the next rank.
Bucket 1 = top 25% (promotion candidates). Small gaps flag salary compression risk.

**Exam Sub-topic:** `dense_rank()`, `ntile()`, `lag()`, `Window.partitionBy().orderBy()`
</details>

<details>
<summary>Scenario 2: Computing Running Revenue Totals and Moving Averages for KPI Dashboard</summary>

**Situation:**
A retail analytics team needs a daily KPI dashboard showing:
1. Cumulative revenue per region year-to-date
2. 7-day moving average to smooth out daily noise
3. Day-over-day revenue change

**Code:**
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, avg, lag, round, col

ytd_window = Window.partitionBy('region', 'year') \
                   .orderBy('sale_date') \
                   .rowsBetween(Window.unboundedPreceding, Window.currentRow)

ma7_window = Window.partitionBy('region') \
                   .orderBy('sale_date') \
                   .rowsBetween(-6, Window.currentRow)  # 7 days inclusive

dod_window = Window.partitionBy('region').orderBy('sale_date')

dashboard = daily_sales \
    .withColumn('ytd_revenue',  sum('revenue').over(ytd_window)) \
    .withColumn('ma7_revenue',  round(avg('revenue').over(ma7_window), 0)) \
    .withColumn('prev_revenue', lag('revenue', 1, 0).over(dod_window)) \
    .withColumn('dod_change',   col('revenue') - col('prev_revenue'))

dashboard.show()
```

**Expected Outcome:**
Daily KPI table with YTD cumulative, 7-day moving average, and day-over-day delta.
All original rows are preserved — no row reduction.

**Exam Sub-topic:** Aggregate window functions; `rowsBetween`; `lag()`; running totals
</details>

<details>
<summary>Scenario 3: Finding Top-N Products per Category for Recommendation Engine</summary>

**Situation:**
An e-commerce platform needs the top 3 products by sales volume within each category
for a 'top picks' recommendation widget. The result must include all product columns
(not just category and count), ruling out `groupBy`.

**Code:**
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, dense_rank, col

w = Window.partitionBy('category').orderBy(col('units_sold').desc())

# Option 1: row_number — always exactly N per category
top3_exact = products \
    .withColumn('rn', row_number().over(w)) \
    .filter(col('rn') <= 3) \
    .drop('rn')

# Option 2: dense_rank — top N ranks (may return >3 rows if ties at boundary)
top3_with_ties = products \
    .withColumn('dr', dense_rank().over(w)) \
    .filter(col('dr') <= 3) \
    .drop('dr')

top3_exact.orderBy('category', col('units_sold').desc()).show()
```

**Expected Outcome:**
Top 3 products per category with all product attributes preserved.
Choice between `row_number` (strict 3) vs `dense_rank` (include ties at rank 3).

**Exam Sub-topic:** Top-N per group pattern; `row_number` vs `dense_rank` for ties
</details>

<details>
<summary>Scenario 4: Detecting Session Boundaries Using lag() for Clickstream Analysis</summary>

**Situation:**
A web analytics team processes clickstream events and needs to identify session boundaries:
a new session starts when the gap between events exceeds 30 minutes.
This is a classic `lag()` use case.

**Code:**
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, unix_timestamp, sum, when

w = Window.partitionBy('user_id').orderBy('event_ts')

# Add previous event timestamp, compute gap, flag new session
sessionized = events \
    .withColumn('prev_ts', lag('event_ts', 1).over(w)) \
    .withColumn('gap_seconds',
        unix_timestamp('event_ts') - unix_timestamp('prev_ts')
    ) \
    .withColumn('is_new_session',
        when(col('prev_ts').isNull() | (col('gap_seconds') > 1800), 1).otherwise(0)
    )

# Running count of session starts = session_id within user
w_cumsum = Window.partitionBy('user_id').orderBy('event_ts') \
                 .rowsBetween(Window.unboundedPreceding, Window.currentRow)

sessionized = sessionized.withColumn(
    'session_id', sum('is_new_session').over(w_cumsum)
)

sessionized.select('user_id', 'event_ts', 'gap_seconds', 'is_new_session', 'session_id').show()
```

**Expected Outcome:**
Each clickstream event has a session ID. Events within 30 minutes of the previous
share the same session ID. The cumulative sum of `is_new_session` provides a clean
monotonically increasing session number per user.

**Exam Sub-topic:** `lag()` for inter-row comparison; cumulative sum as session counter
</details>

<details>
<summary>Scenario 5: Comparing Each Sale to the Previous Year's Same Period with lag()</summary>

**Situation:**
A finance analyst needs a year-over-year (YoY) sales comparison:
for each (region, month), show current year revenue, prior year revenue,
and the YoY growth percentage.

**Code:**
```python
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, round, col, year, month, sum as F_sum

# Aggregate monthly by region
monthly = daily_sales.withColumn('yr', year('sale_date')) \
                     .withColumn('mo', month('sale_date')) \
                     .groupBy('region', 'yr', 'mo') \
                     .agg(F_sum('revenue').alias('monthly_rev'))

# Window ordered by year within (region, month)
yoy_window = Window.partitionBy('region', 'mo').orderBy('yr')

yoy = monthly \
    .withColumn('prev_yr_rev', lag('monthly_rev', 1).over(yoy_window)) \
    .withColumn('yoy_pct',
        round(
            (col('monthly_rev') - col('prev_yr_rev')) / col('prev_yr_rev') * 100,
        1)
    )

yoy.orderBy('region', 'mo', 'yr').show()
```

**Expected Outcome:**
Each (region, month) row shows current revenue, prior year revenue, and YoY% change.
First year rows have NULL for prev_yr_rev (no prior year data).

**Exam Sub-topic:** `lag()` for year-over-year comparison; `Window.partitionBy()` with multiple columns
</details>

## 6. Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| Window vs groupBy | Window keeps all rows; groupBy reduces to N groups |
| `.over(windowSpec)` | Required to make any function a window function |
| `Window` class import | `from pyspark.sql.window import Window` |
| `rank()` ties | 1,1,3 — **skips** rank after tie |
| `dense_rank()` ties | 1,1,2 — **no gap** after tie |
| `row_number()` ties | 1,2,3 — always unique, arbitrary tiebreak |
| `ntile(4)` | Divides partition into 4 equal-size buckets (1=top) |
| `lag(col, 1)` | Value from 1 row **before** — default offset=1, default fill=NULL |
| `lead(col, 1)` | Value from 1 row **after** — default offset=1, default fill=NULL |
| `rowsBetween` | Physical row count |
| `rangeBetween` | Value-based range on ORDER BY column |
| `unboundedPreceding` | `Window.unboundedPreceding` — first row of partition |
| `currentRow` | `Window.currentRow` |
| `unboundedFollowing` | `Window.unboundedFollowing` — last row of partition |
| Running total | `.rowsBetween(Window.unboundedPreceding, Window.currentRow)` |
| 3-row moving avg | `.rowsBetween(-2, Window.currentRow)` |
| Partition total | `.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` |
| ORDER BY required? | Yes for rank/offset functions; optional for pure aggregate windows |
| Top-N per group | `row_number().over(w).filter(col('rn') <= N)` |

---

## 7. Practice Q&A

<details>
<summary>Q1: What is the difference between rank() and dense_rank()?</summary>

**A:** Both assign the same rank to tied values. The difference is what happens to the next rank after a tie:
- `rank()` — **skips** ranks after a tie. Two rows tied at rank 2 → next rank is 4 (rank 3 is skipped).
- `dense_rank()` — **no gaps**. Two rows tied at rank 2 → next rank is 3.

Example with salaries 100k, 90k, 90k, 80k:
```
salary   rank   dense_rank
100000     1        1
 90000     2        2
 90000     2        2      ← both tied
 80000     4        3      ← rank skips 3; dense_rank is consecutive
```
The exam may ask: "Given 5 employees with one tie, what is the maximum rank value?"
- `rank()` max = 5 (if tie is at position 1: 1,1,3,4,5)
- `dense_rank()` max = 4 (if tie is at position 1: 1,1,2,3,4)
</details>

<details>
<summary>Q2: You need the top-1 employee per department. Should you use rank() or row_number()?</summary>

**A:** It depends on how you want to handle ties:
- `row_number() == 1` → always exactly **one** row per department (arbitrary tiebreak when tied)
- `rank() == 1` → all tied top earners get rank 1 → could return **multiple rows** per department
- `dense_rank() == 1` → same as `rank()` for rank=1 specifically

If the exam says "find the **single** top earner per department," use `row_number()`.
If it says "find **all** employees earning the maximum salary per department," use `rank()` or filter on `salary == max(salary).over(window)`.
</details>

<details>
<summary>Q3: What is the difference between rowsBetween and rangeBetween?</summary>

**A:**
- `rowsBetween(start, end)` — frame is defined by **physical row positions**. `-2` means exactly 2 rows before the current row, regardless of the ORDER BY column values.
- `rangeBetween(start, end)` — frame is defined by **value range** of the ORDER BY column. `-100` means all rows where the ORDER BY value is within 100 less than the current row's value.

With `rangeBetween`, rows that have the **same ORDER BY value** (ties) are all included in each other's frame simultaneously — they all get the same result. With `rowsBetween`, each physical row is treated independently.

For running totals, `rowsBetween` is almost always what you want. `rangeBetween` is used for value-based sliding windows (e.g., "all events within 5 minutes of current event").
</details>

<details>
<summary>Q4: What does lag(col, 1, 0) return for the first row in a partition?</summary>

**A:** For the **first row** in a partition, there is no previous row. `lag(col, 1)` returns **NULL** by default. If you specify a third argument as the default value — `lag(col, 1, 0)` — it returns `0` instead of NULL for the first row.

```python
lag('amount', 1)      # first row → NULL
lag('amount', 1, 0)   # first row → 0
lag('amount', 1, -1)  # first row → -1
```

Similarly, `lead(col, 1)` returns NULL for the last row in a partition (no next row exists).
</details>

<details>
<summary>Q5: When do you need a WindowSpec with no partitionBy()?</summary>

**A:** You can define a `WindowSpec` without `partitionBy()` — in that case, the **entire DataFrame is treated as a single partition**. This is valid but means all data must move to a single executor, which can cause severe performance problems on large datasets.

```python
# No partitionBy — global window across ALL rows
w = Window.orderBy('salary')
df.withColumn('global_rank', rank().over(w))  # ranks across entire dataset

# This is a shuffle to a single partition — avoid on large data!
```

Use cases: global ranking (e.g., company-wide salary rank), global running total. Always partition by something when possible to maintain parallelism.
</details>